# 주변 AWS 관측망의 추가 정보와 입력 점검

3목표×(지역4+지역·겨울8)=36적합의 결과와 독립 검산, 23도로·34AWS의 입력 연결을 검토합니다. 풍속은 두 검증을 통과해 별도 실행 모형으로 개선했고, 기온·습도·노면온도·결빙은 유지합니다. 반복 개발 비교이며 공식 성적·새 독립 평가가 아닙니다.

이 노트북은 허용 로컬 자료가 있는 원 작업 공간 또는 v12 모델자료 묶음에서 실행합니다. 원천 입력의 전체 재구축·36모형 재학습은 추가 비공개 자산이 필요합니다. 아래3코드셀은 포함 지표·감사·화면을 검토하고 출력은 저장하지 않습니다.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents]
          if (p/'research/latest/weather_network_20261101_v1/summary.json').exists()
          or (p/'outputs/private/weather_network_20261101_v1/summary.json').exists())
BASE=ROOT/'research/latest' if (ROOT/'research/latest/weather_network_20261101_v1').exists() else ROOT/'outputs/private'
read=lambda p:json.loads(p.read_text(encoding='utf8'))
study=BASE/'weather_network_20261101_v1'
summary=read(study/'summary.json');audit=read(study/'independent_verification.json')
assert len(summary['fits'])==36 and summary['accepted_targets']==['wind_speed']
assert audit['status']=='passed' and len(audit['refits'])==6
assert not audit['protected_outcomes_read'] and not summary['new_independent_assessment']
print('36 comparisons, 6 exact refits, current models preserved')

## 같은 조건의 비교

MAE가 낮을수록 좋습니다. 평균 개선만으로 판단하지 않고 각 겨울·지역·저온·날짜 블록 조건을 함께 확인합니다.

In [ ]:
rows=[]
for target,modes in summary['reports'].items():
    for mode,result in modes.items():
        current=result['metrics']['current']['pooled']['mae']
        candidate=result['metrics']['network_residual']['pooled']['mae']
        assert result['primary_success']==(target=='wind_speed')
        rows.append(dict(target=target,mode=mode,current_mae=current,network_mae=candidate,
                         reduction_pct=100*(1-candidate/current),failed_gates=','.join(k for k,v in result['gates'].items() if not v)))
display(pd.DataFrame(rows))

## 원천 관측과 실제 화면 검증

16예시는 전체 시계열 예측이 아닙니다. 관측소 간 범위와 유효 개수는 입력 진단이며 도로 오차·예측구간·결빙 확률과 구별합니다. HTML을 브라우저에서 열어 도로·시각·변수를 선택하고 원천 CSV를 내려받습니다.

In [ ]:
viewer=BASE/'weather_network_viewer_20261101_v2'
receipt=read(viewer/'summary.json');quality=pd.read_csv(viewer/'network_quality.csv',dtype={'station':str})
sources=pd.read_csv(viewer/'snapshot_sources.csv',dtype={'target':str,'aws':str})
browser=read(BASE/'weather_network_viewer_qa_20261101_v3/summary.json')
visual=read(BASE/'weather_network_viewer_qa_20261101_v3/visual_review.json')
assert (receipt['targets'],receipt['aws_stations'],receipt['snapshot_times'])==(23,34,16)
assert len(sources)==1472 and len(quality)==46 and quality.station.nunique()==23
assert browser['filter_cases']==1840 and browser['javascript_errors']==browser['external_requests']==0
assert visual['status']=='passed' and (viewer/'index.html').is_file()
assert not {'ice','road_temp','state_raw'}&set(sources)
print('Open in Chrome/Edge:',viewer/'index.html')
display(quality.head())